# 🌸 VisionAid - Google Colab Hub

Welcome to the official **VisionAid Google Colab Notebook**!
You can use this notebook for two primary workflows:
1. **🚀 1-Click AI Backend Server (FastAPI + GPU + Cloudflare Public Tunnel)**: Runs YOLO11, EasyOCR & Face Recognition on Colab's free T4 GPU and immediately gives you a public HTTPS URL for your mobile phone.
2. **📱 Android APK Compiler (Buildozer)**: Builds your installable `VisionAid.apk` in a clean Ubuntu Linux environment.

---

## ⚙️ Step 0: Check GPU Status
*(Tip: Go to **Runtime -> Change runtime type** and select **T4 GPU** for 10x faster AI processing)*

In [ ]:
!nvidia-smi

## 📥 Step 1: Clone or Update VisionAid Repository

In [ ]:
import os
if not os.path.exists('VisionAid'):
    !git clone https://github.com/Sonali73724/VisionAid.git
    %cd VisionAid
else:
    %cd VisionAid
    !git pull

!ls -la

--- 
# 🚀 WORKFLOW A: 1-Click AI Backend Server + Public HTTPS Tunnel
Run the two cells below to install dependencies and launch your live AI Server with an instant public URL!

### Step A1: Install Server Dependencies

In [ ]:
!pip install -q fastapi uvicorn ultralytics easyocr opencv-python-headless python-multipart pyngrok

### Step A2: 🚀 1-Click Launch: Server + Cloudflare Public Tunnel
*(Runs YOLO11 & OCR on GPU, launches Cloudflare Tunnel, prints the Public URL, and displays live inference logs)*

In [ ]:
import os
import time
import re
import subprocess
import urllib.request

# 1. Clean up any existing instances
!pkill -f "python server.py" || true
!pkill -f "cloudflared" || true

# 2. Download standalone cloudflared binary if not present
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⬇️ Downloading Cloudflared...")
    !wget -q -nc -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# 3. Start VisionAid Server in the background
print("⏳ Starting VisionAid AI Server on GPU (loading YOLO11, EasyOCR, FaceAnalyzer)...")
server_log = open("server.log", "w")
server_proc = subprocess.Popen(["python", "server.py"], stdout=server_log, stderr=subprocess.STDOUT)

# 4. Wait for server to become healthy
server_ready = False
for i in range(45):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                print("✅ VisionAid Server is UP & HEALTHY on port 8000!")
                break
    except Exception:
        pass

if not server_ready:
    print("⚠️ Server is still initializing or encountered an issue. Latest log:")
    !cat server.log | tail -n 20

# 5. Start Cloudflare Tunnel for port 8000
print("🌐 Opening Cloudflare Public Tunnel...")
cf_log = open("cloudflared.log", "w")
cf_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=cf_log, stderr=subprocess.STDOUT)

# 6. Find public URL in cloudflared log
tunnel_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists("cloudflared.log"):
        text = open("cloudflared.log").read()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    print("\n" + "="*65)
    print("🎉 YOUR PUBLIC VISIONAID SERVER URL:")
    print(f"👉  {tunnel_url}  👈")
    print("="*65)
    print("\n📱 HOW TO CONNECT FROM MOBILE APP:")
    print("1. VisionAid Mobile App me SETTINGS button dabayein.")
    print(f"2. AI SERVER URL field me ye link dalein: {tunnel_url}")
    print("3. 'TEST & SAVE CONNECTION' dabayein -> Green status dot aur voice confirmation aayegi!")
    print("="*65 + "\n")
else:
    print("⚠️ Tunnel output not found yet. Check log below:")
    !cat cloudflared.log | tail -n 20

# 7. Live Stream Server Logs (Press Stop button in Colab to halt)
print("📡 Streaming live AI requests and detections below (Stop button to exit):")
!tail -f server.log

--- 
# 📱 WORKFLOW B: Build Android APK with Buildozer
Run this section if you want to compile your `.apk` installable package.

### Step B1: Install Buildozer & Ubuntu Android Build Tools

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y git zip unzip openjdk-17-jdk autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev cython3
!pip install --upgrade buildozer Cython virtualenv
!buildozer --version

### Step B2: Build the APK (Takes ~10-15 minutes on first run)

In [ ]:
# Compile Android Debug APK
!buildozer -v android debug

### Step B3: Download APK to Your Device

In [ ]:
import glob
from google.colab import files

apk_files = glob.glob('bin/*.apk')
if apk_files:
    print(f"Found APK: {apk_files[0]}")
    print("Downloading APK to your computer/phone...")
    files.download(apk_files[0])
else:
    print("❌ No APK found in bin/ directory. Please check build logs above.")